# Base agent
baseAgent.py allows for easy implementation for AutoGen agents.
This document will explain the purpose of each function and how they should be utilized.
To see this implemented, see adviceAgent.ipynb

## Imports
NOTE: We are using AutoGen version 0.9.10. This is before UserAgent, which has a very different process.

In [ ]:
from autogen import LLMConfig, UserProxyAgent
from typing import List
from langchain.tools import BaseTool
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen.agentchat import ConversableAgent

## Generate LLM Config
This creates the config required for an AutoGen tool.
Without having proper tool config, the agent will not understand their tools and might call it improperly.
This function allows for dynamic tool config generation, meaning that no extra work needs to be done, beside having properly worded tool name, description, and parameters.

In [ ]:
def generate_llm_config(tool: BaseTool):
    args_schema = tool.args_schema.model_json_schema() if hasattr(tool, "args_schema") else {}
    function_schema = {
        "name": tool.name,
        "description": tool.description,
        "parameters": args_schema,
    }
    return {"function": function_schema}

## BaseAgent Class
The main focus when extending this class is to have a proper name, description (which is used for hand offs), system prompt, and tools.
Everything else is done automatically and does not need to be overwritten.

### Get Tools Config
This function gets the config from generate_llm_config for each tool passed in.

### Get Config
This function gets the config for the large language model used for the agent.
For tool execution to work with this setup, this JSON must have:
 - api_type: openai
 - api_key: ollama
 - base_url: http://localhost:11434/v1 (or whatever port ollama is running on instead of 11434)

### Create Agent
This creates an AutoGen Conversable Agent, which is the core of the class. 
We set `human_input_mode = "NEVER"` so that an assistant agent will never prompt the user to reply.
We want this since the orchestrator deals with the user, and after the orchestrator fully understands the user's needs, it will delegate to the assistants.
We also want this because if one of the assistants ask for user input, there will be an error with the web ui API which will crash the program.

### Register Execution
For tool execution to work, two things must be done:
 1. The tool is setup as shown in docs/tools/baseApiTool.ipynb
 2. The user agent 'gives permission' to the assistant agent to run their tool.
This second point is the point of the register execution function.

For example, if an orchestrator wants to have a group conversation with the adviceAgent, the orchestrator file must have the line of code
```adviceAgent.registerExecution(orchestratorAgent)```

In [ ]:
class BaseAgent:
    def __init__(self, name: str, description: str, system_message: str, tools: List[BaseTool]):
        self.name = name
        self.description = description
        self.system_message = system_message
        self.tools = tools
        self.tools_config = self.getToolsConfig()
        self.config = self.getConfig()
        self.agent = self.createAgent()
    
    def getToolsConfig(self):
        tools_config = []
        for tool in self.tools:
            tools_config.append(generate_llm_config(tool))
            
        return tools_config

    def getConfig(self):
        config = LLMConfig(
            tools = self.tools_config,
            config_list = LLMConfig.from_json(path = "OAI_CONFIG_LIST.json").config_list,
            timeout = 120
        )

        return config
    
    def createAgent(self):
        return ConversableAgent(
            name = self.name,
            llm_config = self.config,
            system_message = self.system_message,
            max_consecutive_auto_reply = 3,
            human_input_mode = "NEVER"
        )

    def registerExecution(self, user_proxy: UserProxyAgent):
        for tool in self.tools:
            def make_executor(t):
                return lambda **kwargs: t.invoke(kwargs)
        
            user_proxy.register_for_execution(
                name=tool.name,
            )(make_executor(tool))
